In [1]:
import pandas as pd
import numpy as np

In [2]:
chi_train = pd.read_csv("chicago_2015_2024_shared_final.csv", low_memory=False)
chi_test = pd.read_csv("chicago_2025_shared_final.csv", low_memory=False)
tx_full = pd.read_csv("tx_external_test_panel_final_clean.csv", low_memory=False)

In [3]:
# Check the shape
print("Chicago train shape:", chi_train.shape)
print("Chicago test shape :", chi_test.shape)
print("Texas full shape   :", tx_full.shape)

Chicago train shape: (311140, 54)
Chicago test shape : (34424, 54)
Texas full shape   : (66727, 59)


In [4]:
id_cols_chicago = ["grid_id", "iso_year", "iso_week", "year_week"]
id_cols_texas = ["agency_id", "iso_year", "iso_week", "year_week"]

target_count_cols = [
    "total_crimes",
    "count_THEFT",
    "count_BATTERY",
    "count_CRIMINAL_DAMAGE"
]

label_cols = [
    "label_total_crime",
    "label_THEFT",
    "label_BATTERY",
    "label_CRIMINAL_DAMAGE"
]

shared_feature_cols = [
    # total
    "lag_1w_total", "lag_2w_total", "lag_4w_total", "lag_8w_total", "lag_13w_total", "lag_26w_total", "lag_52w_total",
    "roll_4w_total", "roll_12w_total", "roll_26w_total",

    # THEFT
    "lag_1w_THEFT", "lag_2w_THEFT", "lag_4w_THEFT", "lag_8w_THEFT", "lag_13w_THEFT", "lag_26w_THEFT", "lag_52w_THEFT",
    "roll_4w_THEFT", "roll_12w_THEFT", "roll_26w_THEFT",

    # BATTERY
    "lag_1w_BATTERY", "lag_2w_BATTERY", "lag_4w_BATTERY", "lag_8w_BATTERY", "lag_13w_BATTERY", "lag_26w_BATTERY", "lag_52w_BATTERY",
    "roll_4w_BATTERY", "roll_12w_BATTERY", "roll_26w_BATTERY",

    # CRIMINAL_DAMAGE
    "lag_1w_CRIMINAL_DAMAGE", "lag_2w_CRIMINAL_DAMAGE", "lag_4w_CRIMINAL_DAMAGE", "lag_8w_CRIMINAL_DAMAGE",
    "lag_13w_CRIMINAL_DAMAGE", "lag_26w_CRIMINAL_DAMAGE", "lag_52w_CRIMINAL_DAMAGE",
    "roll_4w_CRIMINAL_DAMAGE", "roll_12w_CRIMINAL_DAMAGE", "roll_26w_CRIMINAL_DAMAGE",

    # time
    "sin_week", "cos_week"
]

print("Shared feature count:", len(shared_feature_cols))

Shared feature count: 42


In [5]:
# Check for duplicate columns
print("Chicago train duplicate cols:", chi_train.columns[chi_train.columns.duplicated()].tolist())
print("Chicago test duplicate cols :", chi_test.columns[chi_test.columns.duplicated()].tolist())
print("Texas full duplicate cols   :", tx_full.columns[tx_full.columns.duplicated()].tolist())

Chicago train duplicate cols: []
Chicago test duplicate cols : []
Texas full duplicate cols   : []


In [6]:
# Check for required columns
required_cols_chicago = id_cols_chicago + target_count_cols + label_cols + shared_feature_cols

missing_in_chi_train = [c for c in required_cols_chicago if c not in chi_train.columns]
missing_in_chi_test = [c for c in required_cols_chicago if c not in chi_test.columns]

print("Missing in Chicago train:", missing_in_chi_train)
print("Missing in Chicago test :", missing_in_chi_test)

Missing in Chicago train: []
Missing in Chicago test : []


In [7]:
required_cols_tx = id_cols_texas + target_count_cols + label_cols + shared_feature_cols

missing_in_tx = [c for c in required_cols_tx if c not in tx_full.columns]
print("Missing in Texas full:", missing_in_tx)

Missing in Texas full: []


In [8]:
# Extract the genuine shared version from Texas
tx_shared = tx_full[id_cols_texas + target_count_cols + label_cols + shared_feature_cols].copy()

print("Texas shared shape:", tx_shared.shape)

Texas shared shape: (66727, 54)


In [9]:
# Extract three feature matrices and directly compare their column names.
X_chi_train = chi_train[shared_feature_cols].copy()
X_chi_test = chi_test[shared_feature_cols].copy()
X_tx = tx_shared[shared_feature_cols].copy()

print("X_chi_train:", X_chi_train.shape)
print("X_chi_test :", X_chi_test.shape)
print("X_tx       :", X_tx.shape)

X_chi_train: (311140, 42)
X_chi_test : (34424, 42)
X_tx       : (66727, 42)


In [10]:
# Verify that column names are exactly the same.
print("Chicago train vs Chicago test columns equal:",
      list(X_chi_train.columns) == list(X_chi_test.columns))

print("Chicago train vs Texas columns equal:",
      list(X_chi_train.columns) == list(X_tx.columns))

print("Chicago test vs Texas columns equal:",
      list(X_chi_test.columns) == list(X_tx.columns))

Chicago train vs Chicago test columns equal: True
Chicago train vs Texas columns equal: True
Chicago test vs Texas columns equal: True


In [11]:
# Verify that the feature data types are consistent.
print("Chicago train dtypes:")
print(X_chi_train.dtypes.value_counts(), "\n")

print("Chicago test dtypes:")
print(X_chi_test.dtypes.value_counts(), "\n")

print("Texas dtypes:")
print(X_tx.dtypes.value_counts(), "\n")

Chicago train dtypes:
float64    42
Name: count, dtype: int64 

Chicago test dtypes:
float64    42
Name: count, dtype: int64 

Texas dtypes:
float64    42
Name: count, dtype: int64 



In [12]:
# Check for missing values
print("Missing values in Chicago train features:")
print(X_chi_train.isna().sum().sum())

print("Missing values in Chicago test features:")
print(X_chi_test.isna().sum().sum())

print("Missing values in Texas features:")
print(X_tx.isna().sum().sum())

Missing values in Chicago train features:
0
Missing values in Chicago test features:
0
Missing values in Texas features:
0


In [13]:
# Does the tag exist?
for df_name, df in {
    "chi_train": chi_train,
    "chi_test": chi_test,
    "tx_shared": tx_shared
}.items():
    print(f"\nChecking labels in {df_name}")
    for col in label_cols:
        print(col, "exists ->", col in df.columns)


Checking labels in chi_train
label_total_crime exists -> True
label_THEFT exists -> True
label_BATTERY exists -> True
label_CRIMINAL_DAMAGE exists -> True

Checking labels in chi_test
label_total_crime exists -> True
label_THEFT exists -> True
label_BATTERY exists -> True
label_CRIMINAL_DAMAGE exists -> True

Checking labels in tx_shared
label_total_crime exists -> True
label_THEFT exists -> True
label_BATTERY exists -> True
label_CRIMINAL_DAMAGE exists -> True


In [14]:
# Are the label values reasonable (must be 0 or 1)?
for df_name, df in {
    "chi_train": chi_train,
    "chi_test": chi_test,
    "tx_shared": tx_shared
}.items():
    print(f"\nLabel unique values in {df_name}")
    for col in label_cols:
        print(col, sorted(df[col].dropna().unique().tolist())[:10])


Label unique values in chi_train
label_total_crime [0, 1]
label_THEFT [0, 1]
label_BATTERY [0, 1]
label_CRIMINAL_DAMAGE [0, 1]

Label unique values in chi_test
label_total_crime [0, 1]
label_THEFT [0, 1]
label_BATTERY [0, 1]
label_CRIMINAL_DAMAGE [0, 1]

Label unique values in tx_shared
label_total_crime [0, 1]
label_THEFT [0, 1]
label_BATTERY [0, 1]
label_CRIMINAL_DAMAGE [0, 1]


In [15]:
# Check the distribution of labels
print("\nChicago train label rates:")
print(chi_train[label_cols].mean())

print("\nChicago test label rates:")
print(chi_test[label_cols].mean())

print("\nTexas label rates:")
print(tx_shared[label_cols].mean())


Chicago train label rates:
label_total_crime        0.859976
label_THEFT              0.571344
label_BATTERY            0.539027
label_CRIMINAL_DAMAGE    0.438478
dtype: float64

Chicago test label rates:
label_total_crime        0.859429
label_THEFT              0.556501
label_BATTERY            0.533175
label_CRIMINAL_DAMAGE    0.420143
dtype: float64

Texas label rates:
label_total_crime        0.686109
label_THEFT              0.448050
label_BATTERY            0.448904
label_CRIMINAL_DAMAGE    0.307027
dtype: float64


In [17]:
non_numeric_chi_train = X_chi_train.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric_chi_test = X_chi_test.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric_tx = X_tx.select_dtypes(exclude=[np.number]).columns.tolist()

print("Non-numeric in Chicago train:", non_numeric_chi_train)
print("Non-numeric in Chicago test :", non_numeric_chi_test)
print("Non-numeric in Texas        :", non_numeric_tx)

print("Missing values in Chicago train features:", X_chi_train.isna().sum().sum())
print("Missing values in Chicago test features :", X_chi_test.isna().sum().sum())
print("Missing values in Texas features        :", X_tx.isna().sum().sum())

schema_ok = (
    list(X_chi_train.columns) == list(X_chi_test.columns) == list(X_tx.columns)
    and len(non_numeric_chi_train) == 0
    and len(non_numeric_chi_test) == 0
    and len(non_numeric_tx) == 0
    and X_chi_train.isna().sum().sum() == 0
    and X_chi_test.isna().sum().sum() == 0
    and X_tx.isna().sum().sum() == 0
)

print("\n=== FINAL CHECK ===")
if schema_ok:
    print("PASS: Chicago and Texas shared datasets are aligned and ready for modeling.")
else:
    print("WARNING: There are still schema/data issues to fix before modeling.")

Non-numeric in Chicago train: []
Non-numeric in Chicago test : []
Non-numeric in Texas        : []
Missing values in Chicago train features: 0
Missing values in Chicago test features : 0
Missing values in Texas features        : 0

=== FINAL CHECK ===
PASS: Chicago and Texas shared datasets are aligned and ready for modeling.


In [18]:
tx_shared.to_csv("data/tx_external_shared_final_42cols.csv", index=False)
X_tx.to_csv("data/tx_external_X_shared_42cols.csv", index=False)

print("Saved:")
print(" - tx_external_shared_final_42cols.csv", tx_shared.shape)
print(" - tx_external_X_shared_42cols.csv", X_tx.shape)

Saved:
 - tx_external_shared_final_42cols.csv (66727, 54)
 - tx_external_X_shared_42cols.csv (66727, 42)
